# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Sau khi chọn Kaggle GPU T4 x2, bật Internet và gắn Road Damage Dataset cùng `thelab10`, dùng Run All để tạo dữ liệu, fine-tune, đánh giá và lưu báo cáo.

## 0. Môi trường (Kaggle Notebook GPU)

In [ ]:
# Kaggle Notebook: Settings → Accelerator → GPU T4 x2; bật Internet.
# Add the Road Damage Dataset as a Kaggle Notebook input before running the data cell.
# Clone the public personal GitHub fork for the instructor scripts.
# Student helpers can be copied from the attached thelab10 Kaggle input.
import os
import subprocess
import shutil
from pathlib import Path

REPO_URL = "https://github.com/vuog23/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_DIR = Path("/kaggle/working/K4-Track-4-Day10-VisionLanguageModel")
if (REPO_DIR / "scripts/prepare_data.py").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
required_files = ("lab_solution.py", "lab_reward.py", "requirements-kaggle.txt")
input_root = Path("/kaggle/input")
for name in required_files:
    target = REPO_DIR / name
    if target.is_file():
        continue
    matches = list(input_root.glob(f"*/{name}"))
    if len(matches) == 1:
        shutil.copy2(matches[0], target)
    elif len(matches) > 1:
        raise RuntimeError(f"Multiple Kaggle inputs provide {name}: {matches}")
missing = [name for name in required_files if not (REPO_DIR / name).is_file()]
if missing:
    raise RuntimeError(f"Missing required lab files in the GitHub clone and Kaggle inputs: {missing}")

!nvidia-smi -L
!pip -q install -r requirements-kaggle.txt
!python -m pytest -q


In [ ]:
# Kaggle input datasets are read-only; working/output files live under /kaggle/working.
import os
import shutil
from pathlib import Path

KAGGLE_OUTPUT = Path("/kaggle/working/day10-output")
KAGGLE_RUNS = KAGGLE_OUTPUT / "checkpoints"
KAGGLE_RUNS.mkdir(parents=True, exist_ok=True)
# Restore checkpoints/evidence from a previously attached Kaggle Notebook output, if present.
for input_root in Path("/kaggle/input").iterdir():
    for saved_dir in input_root.rglob("checkpoints"):
        for saved_run in saved_dir.glob("e[0-9]*"):
            if saved_run.is_dir() and (saved_run / "run.json").exists():
                shutil.copytree(saved_run, KAGGLE_RUNS / saved_run.name, dirs_exist_ok=True)
                shutil.copytree(saved_run, Path("work") / saved_run.name, dirs_exist_ok=True)
for folder in ("evidence", "figures"):
    for input_root in Path("/kaggle/input").iterdir():
        for saved_folder in input_root.rglob(folder):
            if saved_folder.is_dir():
                shutil.copytree(saved_folder, Path("report") / folder, dirs_exist_ok=True)


## 1. Dữ liệu: tải và chia split

In [ ]:
from pathlib import Path
import subprocess
import sys

# Attach lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes with Add Input.
input_root = Path("/kaggle/input")
source_dirs = sorted({p.parent for p in input_root.rglob("images")
                      if p.is_dir() and (p.parent / "labels-YOLO").is_dir()})
if len(source_dirs) != 1:
    raise RuntimeError(f"Expected one attached dataset with images/ and labels-YOLO/; found {source_dirs}")
subprocess.run([sys.executable, "scripts/prepare_data.py", "--source", str(source_dirs[0])], check=True)

In [ ]:
from lab_solution import verify_prepared_data, show_examples

meta = verify_prepared_data()
print(meta["counts"])
show_examples("train", count=6)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json
from lab_solution import load_split, objects_to_answer

sample = load_split("train")[0]
answer = objects_to_answer(sample["objects"], "norm1000", sample["width"], sample["height"])
print("image_id:", sample["image_id"])
print("grounding JSON:", answer)
assert isinstance(json.loads(answer), list)
assert all(0 <= coord <= 1000 for item in json.loads(answer) for coord in item["bbox_2d"])


## 3. E0: zero-shot

In [ ]:
from lab_solution import run_zero_shot

e0_result = run_zero_shot(split="test", batch_size=4)
print("E0 parse_valid_rate:", e0_result["parse_valid_rate"])
print("E0 micro F1:", e0_result["micro"]["f1"], "CI95:", e0_result["f1_ci95"])


In [ ]:
from lab_solution import read_jsonl

# Inspect real raw generations for the E0 discussion in REPORT.md.
for row in read_jsonl("work/e0.jsonl")[:5]:
    print(row["image_id"], "=>", row["raw"])


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from transformers import AutoProcessor
from lab_solution import MODEL_ID, RoadDamageDataset, load_split
import torch

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"
sample_dataset = RoadDamageDataset(load_split("train")[:1], processor)
encoded = sample_dataset[0]
visual_tokens = int(encoded["image_grid_thw"].prod().item())
target_ids = encoded["labels"][encoded["labels"] != -100]
target_text = processor.tokenizer.decode(target_ids.tolist(), skip_special_tokens=False)
print("visual tokens:", visual_tokens)
print("assistant target:", target_text)
assert visual_tokens == 220
assert target_text.rstrip().endswith("<|im_end|>")


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import gc
import torch
from lab_solution import apply_setup, load_model_and_processor

# One-time target/module-count smoke check. E2/E3 are checked again during training.
probe_model, _ = load_model_and_processor(for_training=True)
probe_model, probe_info = apply_setup(probe_model, "E1")
assert probe_info["llm_lora_modules"] == 252
print("E1 trainable parameter count:", probe_info["trainable_params"])
del probe_model
gc.collect()
torch.cuda.empty_cache()


## 6. Vòng huấn luyện

In [ ]:
import json
from pathlib import Path
from lab_solution import train_setup

# Each setup runs 300 optimizer steps. The helper saves adapter/projector snapshots
# to /kaggle/working/day10-output/checkpoints every 100 steps. Completed runs are skipped.
training_results = {}
for setup in ("E1", "E2", "E3"):
    run_name = setup.lower()
    run_meta_path = Path("work") / run_name / "run.json"
    if run_meta_path.exists():
        previous = json.loads(run_meta_path.read_text(encoding="utf-8"))
        if previous.get("completed_steps") == 300:
            print(f"{setup} already has a completed 300-step run; keeping its metadata")
            training_results[setup] = previous
            continue
    training_results[setup] = train_setup(
        setup,
        run_name=run_name,
        backup_dir=KAGGLE_RUNS,
    )


## 7. Suy luận và chấm E1–E3

In [ ]:
import json
import shutil
from pathlib import Path
from lab_solution import evaluate_experiment, render_prediction_examples

# Load E0 from disk if resuming from preserved Kaggle working outputs.
e0_path = Path("report/evidence/e0.json")
if "e0_result" not in globals() and e0_path.exists():
    e0_result = json.loads(e0_path.read_text(encoding="utf-8"))

# Select the E4 baseline by validation F1 so the official test split stays held out.
val_results = {}
for setup in ("E1", "E2", "E3"):
    val_result = evaluate_experiment(setup.lower(), f"{setup.lower()}_val", split="val", batch_size=4)
    val_results[setup] = val_result
    print(f"{setup} validation F1={val_result['micro']['f1']:.4f}, CI95={val_result['f1_ci95']}")
BEST_SETUP = max(("E1", "E2", "E3"), key=lambda setup: val_results[setup]["micro"]["f1"])
print(f"Automatically selected E4 baseline: {BEST_SETUP} (highest validation micro-F1)")

results = {"E0": e0_result}
for setup in ("E1", "E2", "E3"):
    result = evaluate_experiment(setup.lower(), setup.lower(), split="test", batch_size=4)
    results[setup] = result
    print(
        setup,
        "F1=", result["micro"]["f1"],
        "CI95=", result["f1_ci95"],
        "FP/image=", result["fp_per_image"],
        "classes=", {name: row["f1"] for name, row in result["per_class"].items() if name != "other"},
    )

# Copy evidence and annotated figures into Kaggle's notebook output directory.
example_paths = render_prediction_examples("work/e1.jsonl", split="test", count=6)
for folder in ("evidence", "figures"):
    source = Path("report") / folder
    if source.exists():
        shutil.copytree(source, KAGGLE_OUTPUT / folder, dirs_exist_ok=True)
example_paths

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

E4 tự chọn setup có F1 validation cao nhất trong E1–E3, sau đó chạy ablation `data_size`. Có thể dùng **Run All** mà không cần sửa biến thủ công.

In [ ]:
import json
import shutil
from lab_solution import run_e4
from lab_reward import check_iou_reward

# Optional E5 reward unit cases; the GRPO training extension itself is optional.
check_iou_reward()

# Run the required one-axis ablation automatically after validation-based selection.
E4_AXIS = "data_size"
if BEST_SETUP not in {"E1", "E2", "E3"}:
    raise RuntimeError("Run the E1-E3 evaluation cell first; it selects BEST_SETUP automatically")
E4_RESULTS = run_e4(E4_AXIS, BEST_SETUP, backup_dir=KAGGLE_RUNS)
shutil.copytree("report/evidence", KAGGLE_OUTPUT / "evidence", dirs_exist_ok=True)

# Save one compact summary of every measured validation, test, and ablation result.
RESULTS_SUMMARY = {
    "dataset_version": meta["dataset_version"],
    "e4_selection_rule": "highest validation micro-F1 among E1, E2, E3",
    "e4_base_setup": BEST_SETUP,
    "e4_axis": E4_AXIS,
    "validation": val_results,
    "test": results,
    "e4": E4_RESULTS,
}
summary_json = KAGGLE_OUTPUT / "results_summary.json"
summary_json.write_text(json.dumps(RESULTS_SUMMARY, ensure_ascii=False, indent=2), encoding="utf-8")
summary_rows = [(f"{name} val", result) for name, result in val_results.items()]
summary_rows += [(name, result) for name, result in results.items()]
summary_rows += [(item["name"], item) for item in E4_RESULTS]
summary_lines = ["| Run | Split | F1 | CI95 | Parse valid | FP/image |", "|---|---|---:|---:|---:|---:|"]
for name, result in summary_rows:
    ci = result.get("f1_ci95", [0.0, 0.0])
    summary_lines.append(
        f"| {name} | {result.get('split', 'test')} | {result['micro']['f1']:.4f} "
        f"| {ci[0]:.4f}–{ci[1]:.4f} | {result.get('parse_valid_rate', 0.0):.4f} "
        f"| {result.get('fp_per_image', 0.0):.4f} |"
    )
summary_md = KAGGLE_OUTPUT / "results_summary.md"
summary_md.write_text(
    "# Lab Day 10 results\n\n"
    + f"Dataset: `{meta['dataset_version']}`\n\n"
    + f"E4 baseline: **{BEST_SETUP}**, selected by validation micro-F1; axis: **{E4_AXIS}**.\n\n"
    + "\n".join(summary_lines)
    + "\n",
    encoding="utf-8",
)
print(f"Saved complete run summary to {summary_md} and {summary_json}")